# 35 — BoneMarrowWSI tile embeddings (RQ1)

Cuts every fully annotated monolayer ROI into 4 × 4 tiles (about 56 µm) and embeds each tile with the encoder of notebook 03, so that Part I can be run on real tiles with real spatial arrangement (`scripts/36_partI_real_tiles.py`). ROI pixels are read by range requests from the public IDC bucket; whole slides are not downloaded. No labels are used here.

**Inputs:** `rois.csv`, `cells.csv`, `series_index.csv` in `MyDrive/certified-adaptive-pbs/bmwsi_tiles_inputs/`. If absent, cell 0 rebuilds them from the IDC annotations with the parser of `scripts/22_parse_bmwsi.py` and checks them against the repository tables.
**Runtime:** T4 GPU; reading is network-bound (up to about an hour for ~190 ROIs).


In [ ]:
# 0. Setup. If the three input tables are not in Drive, rebuild them (no manual upload needed).
!pip -q install idc-index wsidicom s3fs gcsfs highdicom==0.28.2 timm==1.0.* 2>/dev/null
import os, json, time, hashlib, platform, glob, shutil, zipfile
import numpy as np, pandas as pd, torch, timm
from google.colab import drive; drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/certified-adaptive-pbs'; INP = f'{DRIVE}/bmwsi_tiles_inputs'; OUT = f'{DRIVE}/bmwsi_tiles_v1'
os.makedirs(OUT, exist_ok=True); os.makedirs(INP, exist_ok=True)

def parse_bmwsi(D):
    """Identical to scripts/22_parse_bmwsi.py (verified: reproduces data/bmwsi/cells.csv and rois.csv exactly, highdicom 0.28.2)."""
    import pydicom, highdicom as hd
    cells, rois = [], []
    for f in sorted(glob.glob(f"{D}/bonemarrowwsi_pediatricleukemia/**/*.dcm", recursive=True)):
        ds = pydicom.dcmread(f); desc = ds.SeriesDescription; pid = ds.PatientID
        ann = hd.ann.MicroscopyBulkSimpleAnnotations.from_dataset(ds)
        src = "roi" if desc.startswith("Monolayer") else ("consensus" if desc.startswith("Consensus") else "unlabeled")
        for g in ann.get_annotation_groups():
            for c in g.get_graphic_data(coordinate_type=ds.AnnotationCoordinateType):
                c = np.asarray(c, float); x0, y0 = c[:, 0].min(), c[:, 1].min(); x1, y1 = c[:, 0].max(), c[:, 1].max()
                row = dict(patient=pid, series=ds.SeriesInstanceUID, x0=x0, y0=y0, x1=x1, y1=y1)
                if src == "roi": rois.append(row)
                else: cells.append(dict(row, source=src, label=g.label, meaning=g.annotated_property_type.meaning,
                                        cx=(x0 + x1) / 2, cy=(y0 + y1) / 2))
    C = pd.DataFrame(cells); R = pd.DataFrame(rois)
    R["roi"] = R.groupby("patient").cumcount()
    C["roi"] = -1
    for pid, g in R.groupby("patient"):
        m = C.patient == pid
        for _, r in g.iterrows():
            inside = m & (C.cx >= r.x0) & (C.cx <= r.x1) & (C.cy >= r.y0) & (C.cy <= r.y1)
            C.loc[inside, "roi"] = r.roi
    return C, R

def fingerprint(R, C):
    a = R[['patient','roi','x0','y0','x1','y1']].astype({'x0':int,'y0':int,'x1':int,'y1':int}).sort_values(['patient','roi']).to_csv(index=False)
    b = C[['patient','roi','source','label','x0','y0','x1','y1']].astype({'x0':int,'y0':int,'x1':int,'y1':int}) \
          .sort_values(['patient','source','x0','y0','x1','y1','label']).to_csv(index=False)
    return hashlib.sha256(a.encode()).hexdigest()[:16], hashlib.sha256(b.encode()).hexdigest()[:16]
EXPECTED = ('23d6a9d242d1e4a1', 'e73ccda91a271f29')   # data/bmwsi/rois.csv, cells.csv in the repo (810 ROIs, 44,741 boxes)

need = [f for f in ['rois.csv', 'cells.csv', 'series_index.csv'] if not os.path.exists(f'{INP}/{f}')]
if need:
    print('missing in Drive:', need, '-> rebuilding the input tables')
    ANN = '/content/bm_ann'; shutil.rmtree(ANN, ignore_errors=True)
    zp = f'{DRIVE}/bmwsi_ann.zip'
    if os.path.exists(zp):
        zipfile.ZipFile(zp).extractall(ANN); print('  source:', zp, '(from notebook 21)')
    else:
        from idc_index import IDCClient
        c = IDCClient(); d = c.index
        keep = d[(d.collection_id == 'bonemarrowwsi_pediatricleukemia') & (d.Modality == 'ANN') &
                 d.SeriesDescription.isin(['Monolayer regions of interest for cell classification',
                                           'Consensus: cell bounding boxes with cell type labels',
                                           'Unlabeled cell bounding boxes'])]
        c.download_from_selection(seriesInstanceUID=list(keep.SeriesInstanceUID), downloadDir=ANN)
        keep[['PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SeriesDescription', 'series_size_MB']].to_csv(
            f'{ANN}/series_index.csv', index=False)
        print('  source: IDC download,', len(keep), 'annotation series')
    C_, R_ = parse_bmwsi(ANN)
    C_.to_csv(f'{INP}/cells.csv', index=False); R_.to_csv(f'{INP}/rois.csv', index=False)
    shutil.copy(f'{ANN}/series_index.csv', f'{INP}/series_index.csv')
    print('  rebuilt:', len(R_), 'ROIs,', len(C_), 'boxes')
fp = fingerprint(pd.read_csv(f'{INP}/rois.csv'), pd.read_csv(f'{INP}/cells.csv'))
print('fingerprint', fp, '| matches repo tables:', fp == EXPECTED)
if fp != EXPECTED:
    print('WARNING: these tables differ from the repo version used in the paper. Tiles may not line up with the labels '
          'that scripts/36 reads. Resolve this before going on.')
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())


In [ ]:
# 1. Eligible ROIs (same rules as scripts/23_bmwsi_clustering.py, but ROIs WITHOUT blasts are kept: Part I needs them)
EXCL = {'artifact','damaged_cell','smudge_cell','technically_unfit','thrombocyte','thrombocyte_aggregate','giant_platelet',
        'erythrocyte','spicule','neutrophil_extracellular_trap','no_consensus_found','megakaryocyte','micromegakaryocyte',
        'promegakaryocyte','mitosis','phagocytosis','pseudo_gaucher_cell','macrophage'}
R = pd.read_csv(f'{INP}/rois.csv'); C = pd.read_csv(f'{INP}/cells.csv'); SI = pd.read_csv(f'{INP}/series_index.csv')
R = R[((R.x1 - R.x0).between(1500, 3000)) & ((R.y1 - R.y0).between(1500, 3000))]
Cn = C[(C.source == 'consensus') & (C.roi >= 0) & (~C.label.isin(EXCL))]
n_nuc = Cn.groupby(['patient', 'roi']).size().rename('n_nuc').reset_index()
R = R.merge(n_nuc, on=['patient', 'roi']); R = R[R.n_nuc >= 20].copy()
R = R.merge(SI[['SeriesInstanceUID', 'StudyInstanceUID']], left_on='series', right_on='SeriesInstanceUID', how='left')
from idc_index import IDCClient
idc = IDCClient()
sm = idc.sql_query("SELECT StudyInstanceUID, SeriesInstanceUID AS sm_series FROM index "
                   "WHERE collection_id='bonemarrowwsi_pediatricleukemia' AND Modality='SM'")
R = R.merge(sm, on='StudyInstanceUID', how='left')
print(len(R), 'ROIs |', R.patient.nunique(), 'patients | mapped to a slide:', R.sm_series.notna().sum())
assert R.sm_series.notna().all()

In [ ]:
# 2. Open a slide lazily over the network and read one ROI at full resolution (level 0)
from wsidicom import WsiDicom
def open_slide(series_uid):
    urls = idc.get_series_file_URLs(series_uid)             # s3://idc-open-data/<uuid>/<uuid>.dcm, one per pyramid level
    try:
        return WsiDicom.open(urls, file_options={'anon': True})
    except Exception as e:                                     # fall back to the Google Cloud mirror
        print('  s3 open failed, trying gs:', repr(e)[:200])
        gs = [u.replace('s3://', 'gs://') for u in urls]
        return WsiDicom.open(gs, file_options={'token': 'anon'})
r0 = R.iloc[0]; t0 = time.time(); sl = open_slide(r0.sm_series)
base = sl.levels.base_level
print('base level size:', base.size, '| mpp:', base.mpp, f'| open {time.time()-t0:.1f}s')
im = sl.read_region((int(r0.x0), int(r0.y0)), 0, (int(r0.x1 - r0.x0), int(r0.y1 - r0.y0)))
print('ROI image', im.size, f'| read {time.time()-t0:.1f}s'); im.resize((512, 512)).save(f'{OUT}/check_roi0.jpg')
# CHECK the saved check_roi0.jpg: it must show a blood-cell monolayer. If it shows glass or the wrong place, the
# annotation coordinates are not level-0 pixel coordinates; stop and record the printed base-level size and mpp.

In [ ]:
# 3. Encoder, identical to 03_embeddings_colab.ipynb
SEED = 20260919; torch.manual_seed(SEED); np.random.seed(SEED)
MODEL_NAME, IMG_SIZE = 'vit_small_patch14_dinov2.lvd142m', 224
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0, img_size=IMG_SIZE).eval().to(device)
cfg = timm.data.resolve_data_config({}, model=model); MEAN, STD = cfg['mean'], cfg['std']
import torchvision.transforms.functional as TF
def embed(imgs):
    x = torch.stack([TF.normalize(TF.to_tensor(TF.resize(i.convert('RGB'), [IMG_SIZE, IMG_SIZE], antialias=True)), MEAN, STD) for i in imgs])
    with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
        return model(x.to(device)).float().cpu().numpy()

In [ ]:
# 4. Tile every ROI 4 x 4 and embed (resumable: one .npz per ROI in OUT/parts)
G = 4; os.makedirs(f'{OUT}/parts', exist_ok=True); t0 = time.time()
for sid, Rs in R.groupby('sm_series'):
    todo = [r for _, r in Rs.iterrows() if not os.path.exists(f"{OUT}/parts/{r.patient}_{r.roi}.npz")]
    if not todo: continue
    try: sl = open_slide(sid)
    except Exception as e: print('SKIP slide', sid, repr(e)[:200]); continue
    for r in todo:
        w, h = int(r.x1 - r.x0), int(r.y1 - r.y0)
        try: im = sl.read_region((int(r.x0), int(r.y0)), 0, (w, h))
        except Exception as e: print('SKIP roi', r.patient, r.roi, repr(e)[:200]); continue
        tiles, boxes = [], []
        for i in range(G):          # i = column (x), j = row (y); tile index i*G + j matches scripts/23 and 30
            for j in range(G):
                a0, b0 = i * w // G, j * h // G; a1, b1 = (i + 1) * w // G, (j + 1) * h // G
                tiles.append(im.crop((a0, b0, a1, b1))); boxes.append((r.x0 + a0, r.y0 + b0, r.x0 + a1, r.y0 + b1))
        np.savez(f"{OUT}/parts/{r.patient}_{r.roi}.npz", emb=embed(tiles).astype(np.float16), boxes=np.array(boxes),
                 tile=np.arange(G * G), patient=r.patient, roi=r.roi)
    print(f'{sid[-12:]}: {len(todo)} ROIs | {time.time()-t0:.0f}s', flush=True)

In [ ]:
# 5. Collect, save, record config
rows, E = [], []
for f in sorted(os.listdir(f'{OUT}/parts')):
    z = np.load(f'{OUT}/parts/{f}', allow_pickle=True)
    for t, b in zip(z['tile'], z['boxes']):
        rows.append(dict(patient=str(z['patient']), roi=int(z['roi']), tile=int(t), x0=b[0], y0=b[1], x1=b[2], y1=b[3]))
    E.append(z['emb'])
T = pd.DataFrame(rows); E = np.concatenate(E)
T.to_csv(f'{OUT}/tiles.csv', index=False); np.save(f'{OUT}/tiles_embeddings_dinov2_vits14.npy', E)
json.dump(dict(notebook='35_bmwsi_tiles_colab.ipynb', created_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
               model=MODEL_NAME, img_size=IMG_SIZE, grid=G, n_rois=int(T.groupby(['patient','roi']).ngroups), n_tiles=len(T),
               torch=torch.__version__, timm=timm.__version__, python=platform.python_version(),
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'),
          open(f'{OUT}/config.json', 'w'), indent=1)
print(len(T), 'tiles from', T.patient.nunique(), 'patients ->', OUT)

## Output
`bmwsi_tiles_v1/` → `data/bmwsi_tiles/`, then `scripts/36_partI_real_tiles.py`.
